# DSA 4060 — Week 9 Guided Lab
## From implicit events to ranked recommendations with confidence-weighted ALS

**Format:** instructor-led code-along · **Time:** 90–120 min · **Environment:** Jupyter in VS Code
**Output:** a trained ALS model, Top-5 recommendations, and a short reflection

**Learning goals**
- Explain why views, saves, plays and purchases are useful but ambiguous signals.
- Convert event logs into interaction strength, preference and confidence.
- Represent interactions in a sparse user–item matrix.
- Train an implicit-feedback Alternating Least Squares (ALS) model.
- Produce and critique a ranked Top-N list.

> Work in pairs: one **driver**, one **reviewer**. Switch roles halfway through.
> Run cells in order. Pause at every **Checkpoint**.

## 1. Set up the notebook
`%pip` installs into the *active kernel*. If VS Code asks, restart the kernel and rerun this cell.

In [1]:
%pip install implicit pandas numpy scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 38.3 MB/s  0:00:00


In [2]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares

np.random.seed(4060)

## 2. Create a small event log
A compact music-streaming dataset so everyone starts from the same data. Each row is an observed **event**, not a rating.

In [3]:
users = [f"U{i:02d}" for i in range(1, 13)]
items = [f"Track_{i:02d}" for i in range(1, 19)]
event_types = ["play", "repeat_play", "save"]

rows = []
for user in users:
    preferred = np.random.choice(items, size=6, replace=False)
    for item in preferred:
        n = np.random.randint(1, 5)
        for _ in range(n):
            rows.append({
                "user_id": user,
                "item_id": item,
                "event_type": np.random.choice(event_types, p=[0.60, 0.25, 0.15])
            })

events = pd.DataFrame(rows)
events.head(), events.shape

(  user_id   item_id   event_type
 0     U01  Track_02  repeat_play
 1     U01  Track_02  repeat_play
 2     U01  Track_02         play
 3     U01  Track_02         save
 4     U01  Track_06  repeat_play,
 (194, 3))

In [4]:
rng = np.random.default_rng(4060)
start = pd.Timestamp("2026-10-01")
events["timestamp"] = start + pd.to_timedelta(rng.integers(0, 30 * 24 * 60, size=len(events)), unit="m")
events.head()

,user_id,item_id,event_type,timestamp
0,U01,Track_02,repeat_play,2026-10-02 04:12:00
1,U01,Track_02,repeat_play,2026-10-02 00:20:00
2,U01,Track_02,play,2026-10-22 06:56:00
3,U01,Track_02,save,2026-10-20 23:52:00
4,U01,Track_06,repeat_play,2026-10-23 22:00:00


Quick inspection:

In [5]:
display(events['event_type'].value_counts())
display(events.groupby('user_id').size().describe())

,count
event_type,
play,120
repeat_play,44
save,30


,0
count,12.000000
mean,16.166667
std,3.511885
min,12.000000
25%,13.750000
50%,15.000000
75%,17.750000
max,23.000000


## 3. Turn events into interaction strength
Larger weights for actions that usually indicate stronger interest. These weights are **modeling choices, not universal truths**.

In [6]:
event_weight = {
    "play": 1.0,
    "repeat_play": 3.0,
    "save": 5.0
}

events['weight'] = events['event_type'].map(event_weight)

interactions = (events
    .groupby(['user_id', 'item_id'], as_index=False)['weight']
    .sum()
    .rename(columns={'weight': 'strength'}))

interactions.sort_values('strength', ascending=False).head(10)

,user_id,item_id,strength
1,U01,Track_02,12.0
71,U12,Track_12,12.0
61,U11,Track_03,12.0
69,U12,Track_09,12.0
14,U03,Track_11,11.0
70,U12,Track_10,11.0
8,U02,Track_10,10.0
11,U02,Track_17,10.0
67,U12,Track_05,10.0
54,U10,Track_06,10.0


## 4. Preference and confidence
For implicit feedback, **preference** is binary (was an interaction observed?). **Confidence** grows with observed strength: $c_{ui} = 1 + \alpha \cdot r_{ui}$.

In [7]:
alpha = 20

interactions['preference'] = (interactions['strength'] > 0).astype(int)
interactions['confidence'] = 1 + alpha * interactions['strength']

interactions[['user_id', 'item_id', 'strength', 'preference', 'confidence']].head()

,user_id,item_id,strength,preference,confidence
0,U01,Track_01,2.0,1,41.0
1,U01,Track_02,12.0,1,241.0
2,U01,Track_06,6.0,1,121.0
3,U01,Track_08,3.0,1,61.0
4,U01,Track_11,8.0,1,161.0


*Preference* answers "did we observe positive behavior?" *Confidence* answers "how strongly should the model trust that observation?"

### Checkpoint 3 — Compare two cases
Run the cell below, then discuss.

In [8]:
for s in (1, 12):
    print(f"strength={s:>2} -> preference=1, confidence={1 + alpha * s}")
print("unobserved pair -> preference=0, confidence=1 (baseline)")

strength= 1 -> preference=1, confidence=21
strength=12 -> preference=1, confidence=241
unobserved pair -> preference=0, confidence=1 (baseline)


## 5. Build the sparse user–item matrix

In [9]:
user_ids = sorted(interactions['user_id'].unique())
item_ids = sorted(interactions['item_id'].unique())

user_to_idx = {u: i for i, u in enumerate(user_ids)}
item_to_idx = {it: i for i, it in enumerate(item_ids)}
idx_to_item = {i: it for it, i in item_to_idx.items()}

row = interactions['user_id'].map(user_to_idx)
col = interactions['item_id'].map(item_to_idx)
data = interactions['strength'].astype(float)

user_item = csr_matrix((data, (row, col)), shape=(len(user_ids), len(item_ids)))

print('shape:', user_item.shape)
print('observed interactions:', user_item.nnz)
print('density:', user_item.nnz / (user_item.shape[0] * user_item.shape[1]))

shape: (12, 18)
observed interactions: 72
density: 0.3333333333333333


## 6. Train an ALS model
ALS alternates between learning user factors and item factors, placing users and items in the same latent space.

In [10]:
model = AlternatingLeastSquares(
    factors=16,
    regularization=0.10,
    alpha=alpha,
    iterations=25,
    random_state=4060
)
model.fit(user_item)

print(model.user_factors.shape, model.item_factors.shape)

/usr/local/lib/python3.13/dist-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/25 [00:00<?, ?it/s]

(12, 16) (18, 16)


## 7. Generate Top-5 recommendations

In [11]:
target_user = 'U01'
u = user_to_idx[target_user]

item_idx, scores = model.recommend(
    u,
    user_item[u],
    N=5,
    filter_already_liked_items=True
)

recommendations = pd.DataFrame({
    'item_id': [idx_to_item[i] for i in item_idx],
    'score': scores
})

seen = (interactions.loc[interactions['user_id'].eq(target_user), ['item_id', 'strength']]
        .sort_values('strength', ascending=False))

print('Observed history')
display(seen.head(10))
print('Recommended next')
display(recommendations)

Observed history


,item_id,strength
1,Track_02,12.0
4,Track_11,8.0
5,Track_13,6.0
2,Track_06,6.0
3,Track_08,3.0
0,Track_01,2.0


Recommended next


,item_id,score
0,Track_05,0.186877
1,Track_04,0.125515
2,Track_09,0.093726
3,Track_07,0.069388
4,Track_12,0.063325


## 8. One-variable experiment
Change **only one** setting, retrain, and compare the Top-5 list for U01. Each pair picks one experiment.

| Experiment | Values to compare | Question |
|---|---|---|
| Confidence scale | `alpha` = 5 vs. 40 | Does stronger confidence change the ranking? |
| Latent dimension | `factors` = 4 vs. 32 | Does more capacity produce a different list? |
| Regularization | 0.01 vs. 1.0 | Does stronger shrinkage stabilize or flatten scores? |

The helper below retrains with the baseline settings, overriding only what you pass in.

In [12]:
BASE = dict(factors=16, regularization=0.10, alpha=20, iterations=25, random_state=4060)

def train_and_recommend(user='U01', N=5, matrix=None, **overrides):
    params = {**BASE, **overrides}
    m = AlternatingLeastSquares(**params)
    mat = user_item if matrix is None else matrix
    m.fit(mat, show_progress=False)
    uidx = user_to_idx[user]
    idx, sc = m.recommend(uidx, mat[uidx], N=N, filter_already_liked_items=True)
    return pd.DataFrame({'item_id': [idx_to_item[i] for i in idx], 'score': sc})

# --- CHOOSE ONE experiment: uncomment exactly one line ---
experiment = {'alpha': 5}, {'alpha': 40}
# experiment = {'factors': 4}, {'factors': 32}
# experiment = {'regularization': 0.01}, {'regularization': 1.0}

low = train_and_recommend(**experiment[0])
high = train_and_recommend(**experiment[1])

comparison = pd.DataFrame({
    'baseline_item': recommendations['item_id'].values,
    f'{list(experiment[0])[0]}={list(experiment[0].values())[0]}': low['item_id'].values,
    f'{list(experiment[1])[0]}={list(experiment[1].values())[0]}': high['item_id'].values,
})
display(comparison)

print("Overlap with baseline Top-5:",
      len(set(low['item_id']) & set(recommendations['item_id'])), "/5 (low),",
      len(set(high['item_id']) & set(recommendations['item_id'])), "/5 (high)")
print("Score range  low :", low['score'].min().round(3), "to", low['score'].max().round(3))
print("Score range  high:", high['score'].min().round(3), "to", high['score'].max().round(3))

,baseline_item,alpha=5,alpha=40
0,Track_05,Track_12,Track_12
1,Track_04,Track_05,Track_10
2,Track_09,Track_07,Track_04
3,Track_07,Track_09,Track_18
4,Track_12,Track_04,Track_07


Overlap with baseline Top-5: 5 /5 (low), 3 /5 (high)


AttributeError: 'float' object has no attribute 'round'

## 9. Evaluation: design before calculating
A model that fits observed events isn't automatically useful. A realistic evaluation **hides later interactions** and asks whether the model ranks them highly.

1. Sort each user's events by time.
2. Train on earlier interactions.
3. Hold out a later interaction as the relevant test item.
4. Recommend Top-K without already-seen items.
5. Compute Precision@K / Recall@K / NDCG@K and compare with a **popularity baseline**.

Here we hold out each user's *most recent item* (by first-seen time among items) and train on everything else.

In [13]:
# Time-aware leave-last-item-out split
ev = events.sort_values('timestamp')
first_seen = ev.groupby(['user_id', 'item_id'], as_index=False)['timestamp'].min()
last_item = (first_seen.sort_values('timestamp').groupby('user_id').tail(1)
             [['user_id', 'item_id']].assign(held_out=True))

train_ev = ev.merge(last_item, on=['user_id', 'item_id'], how='left')
train_ev = train_ev[train_ev['held_out'].isna()]

train_int = (train_ev.groupby(['user_id', 'item_id'], as_index=False)['weight'].sum()
             .rename(columns={'weight': 'strength'}))

# Reuse the same index maps so shapes line up
train_matrix = csr_matrix(
    (train_int['strength'].astype(float),
     (train_int['user_id'].map(user_to_idx), train_int['item_id'].map(item_to_idx))),
    shape=user_item.shape)

test_items = {r.user_id: item_to_idx[r.item_id] for r in last_item.itertuples()}
print("train interactions:", train_matrix.nnz, "| held-out:", len(test_items))

train interactions: 60 | held-out: 12


In [14]:
K = 5

def recall_ndcg_at_k(rec_lists, test_items, k=K):
    recalls, ndcgs = [], []
    for user, target in test_items.items():
        recs = list(rec_lists[user][:k])
        hit = target in recs
        recalls.append(1.0 if hit else 0.0)          # one held-out item => recall = hit rate
        ndcgs.append(1.0 / np.log2(recs.index(target) + 2) if hit else 0.0)
    return np.mean(recalls), np.mean(ndcgs)

# ALS recommendations from the training-only matrix
als_eval = AlternatingLeastSquares(**BASE)
als_eval.fit(train_matrix, show_progress=False)
als_recs = {}
for user in user_ids:
    uidx = user_to_idx[user]
    idx, _ = als_eval.recommend(uidx, train_matrix[uidx], N=K, filter_already_liked_items=True)
    als_recs[user] = idx

# Popularity baseline: most-interacted-with items, excluding each user's seen items
pop_order = np.asarray((train_matrix > 0).sum(axis=0)).ravel().argsort()[::-1]
pop_recs = {}
for user in user_ids:
    seen_idx = set(train_matrix[user_to_idx[user]].indices)
    pop_recs[user] = [i for i in pop_order if i not in seen_idx][:K]

results = pd.DataFrame(
    [recall_ndcg_at_k(als_recs, test_items), recall_ndcg_at_k(pop_recs, test_items)],
    index=['ALS', 'Popularity'], columns=[f'Recall@{K}', f'NDCG@{K}'])
results

,Recall@5,NDCG@5
ALS,0.250000,0.126482
Popularity,0.416667,0.249357


>  With only 12 users and one held-out item each, this result is **extremely noisy** (each user moves Recall@5 by ~0.08). Treat it as a demonstration of the *procedure*, not evidence that one model is better. The data is also random, so there's no real taste structure for ALS to learn.



## 10. Whole-class debrief

| Concept | Where it appeared | In your own words |
|---|---|---|
| Implicit feedback | Event log and strength | Observed behavior is useful but not the same as a rating. |
| Confidence weighting | `alpha` and interaction strength | Repeated/stronger signals receive greater influence. |
| Sparse matrix | CSR user–item matrix | We store observed entries efficiently; most pairs are missing. |
| ALS | Alternating latent-factor training | The model learns compact user and item representations. |
| Ranking | Top-5 output | The practical goal is an ordered shortlist, not just a score. |
| Evaluation | Time-aware holdout | We test whether future relevant items appear near the top. |

In [15]:
# Optional: word counter. Paste your response between the quotes and run.
response = """
"""
print(len(response.split()), "words (target: 150-200)")

0 words (target: 150-200)


In [16]:
# Starter: catalog coverage and popularity bias of ALS Top-5 across all users
all_recs = []
for user in user_ids:
    uidx = user_to_idx[user]
    idx, _ = model.recommend(uidx, user_item[uidx], N=5, filter_already_liked_items=True)
    all_recs.extend(idx)

rec_counts = pd.Series([idx_to_item[i] for i in all_recs]).value_counts()
print(f"Coverage: {rec_counts.shape[0]} of {len(item_ids)} items recommended at least once "
      f"({rec_counts.shape[0] / len(item_ids):.0%})")
display(rec_counts.head(5).rename('times_recommended'))

Coverage: 17 of 18 items recommended at least once (94%)


,times_recommended
Track_09,5
Track_11,5
Track_07,5
Track_15,4
Track_04,4


## Troubleshooting
| Symptom | Try this |
|---|---|
| Import error after installation | Confirm the VS Code kernel, run `%pip install` in a cell, restart the kernel, rerun cells in order. |
| Empty or repeated recommendations | Confirm `N` is smaller than the catalog and `filter_already_liked_items=True`. |
| Index mismatch | Rebuild `user_to_idx`, `item_to_idx`, and the sparse matrix in the same run. |
| Cell will not run | Install/enable the Microsoft Python and Jupyter extensions, then Select Kernel. |
| Results differ across runs | Keep `random_state` and the NumPy seed fixed. |